# Ayso 3D — صورة ← مجسّم (TripoSR)

**قبل أي شي: Runtime → Change runtime type → T4 GPU → Save**

## الترتيب
١ · ٢ · ٣ · ٤ — **بلا restart، بلا انقطاع.**

> ما نغيّر نسخ المكتبات — نصلّح TripoSR ليتوافق مع بيئة كولاب الحالية.

In [ ]:
# ═══════ الخلية ١ — تجهيز (٣ دقايق) ═══════
# ملاحظة: ما نغيّر نسخة transformers — كولاب على Python 3.13
# و transformers 4.44 ما يشتغل عليه. نصلّح TripoSR ليتوافق مع 5.x.
import sys, subprocess, os

print("تثبيت المكتبات…")
subprocess.run([sys.executable,"-m","pip","install","-q",
    "omegaconf","einops","trimesh","rembg","onnxruntime","scikit-image",
    "huggingface_hub"], check=False)
print("  ✓")

print("تنزيل TripoSR…")
if not os.path.exists("/content/TripoSR"):
    subprocess.run("cd /content && git clone --depth 1 https://github.com/VAST-AI-Research/TripoSR.git",
                   shell=True, capture_output=True)
print("  ✓")

# بديل torchmcubes (الأصلي يحتاج ترجمة CUDA → يفشل بكولاب)
SHIM = """
import torch, numpy as np
from skimage import measure
def marching_cubes(vol, thresh, **kw):
    v = np.asarray(vol.detach().cpu().numpy() if hasattr(vol, "detach") else vol)
    if v.ndim == 4: v = v[0]
    v = v.astype(np.float32)
    if not np.isfinite(v).all() or v.min() >= thresh or v.max() <= thresh:
        return torch.zeros((0,3), dtype=torch.float32), torch.zeros((0,3), dtype=torch.int64)
    verts, faces, _, _ = measure.marching_cubes(v, level=float(thresh))
    return torch.from_numpy(verts.astype(np.float32)), torch.from_numpy(faces.astype(np.int64))
"""
open("/content/torchmcubes.py","w").write(SHIM)
print("  ✓ بديل torchmcubes")

import torch, transformers
print(f"\n  torch {torch.__version__} · transformers {transformers.__version__}")
print(f"  كارت: {'✓' if torch.cuda.is_available() else '✗ — بدّل لـT4'}")
print("\n✅ الخلية ١ تامّة — روح للخلية ٢ (بلا restart)")

In [ ]:
# ═══════ الخلية ٢ — تحميل النموذج + رفع الصورة ═══════
import sys, os
sys.path.insert(0, "/content/TripoSR")
if "/content" not in sys.path: sys.path.insert(0, "/content")
import importlib; importlib.invalidate_caches()

import torch, transformers
assert torch.cuda.is_available(), "فعّل T4: Runtime → Change runtime type → T4 GPU"
import torchmcubes  # البديل
from tsr.system import TSR

# ── إعادة تسمية مفاتيح الشيكبوينت (صيغة transformers 4.x → 5.x) ──
REMAPS = [
    (".encoder.layer.", ".layers."),
    (".attention.attention.query", ".attention.q_proj"),
    (".attention.attention.key",   ".attention.k_proj"),
    (".attention.attention.value", ".attention.v_proj"),
    (".attention.output.dense",    ".attention.o_proj"),
    (".intermediate.dense",        ".mlp.fc1"),
    (".output.dense",              ".mlp.fc2"),
]
def _remap(k):
    for a, b in REMAPS: k = k.replace(a, b)
    return k

_orig = torch.load
def _patched(*a, **kw):
    try:
        sd = _orig(*a, **kw)
    except Exception:
        kw["weights_only"] = False
        sd = _orig(*a, **kw)
    if isinstance(sd, dict):
        k0 = next(iter(sd), "")
        if isinstance(k0, str) and ".encoder.layer." in k0:
            sd = {_remap(k): v for k, v in sd.items()}
            print("   مفاتيح مصلَّحة ✓")
    return sd

print("تحميل النموذج (١.٧ غ)…")
torch.load = _patched
# نمنع الـstrict load: نبني النموذج وندخل الأوزان بأنفسنا
from omegaconf import OmegaConf
from huggingface_hub import hf_hub_download
cfg = OmegaConf.load(hf_hub_download("stabilityai/TripoSR", "config.yaml"))
OmegaConf.resolve(cfg)
wgt = hf_hub_download("stabilityai/TripoSR", "model.ckpt")
MODEL = TSR(cfg)
sd = _patched(wgt, map_location="cpu")
torch.load = _orig
missing, unexpected = MODEL.load_state_dict(sd, strict=False)
print(f"   ناقصة: {len(missing)} · زايدة: {len(unexpected)}")

crit = [k for k in missing if any(s in k for s in ("q_proj","k_proj","v_proj","o_proj","mlp.fc1","mlp.fc2"))]
assert not crit, f"⚠️ أوزان أساسية ناقصة: {crit[:3]}"

MODEL.renderer.set_chunk_size(32768)
MODEL.to("cuda"); MODEL.eval()
print("✓ النموذج جاهز\n")

from google.colab import files
up = files.upload()
IMAGE = "/content/" + list(up.keys())[0]
from PIL import Image
im = Image.open(IMAGE)
print("الصورة:", IMAGE, im.size)
display(im.convert("RGB").resize((300, int(300*im.size[1]/im.size[0]))))
print("\n✅ روح للخلية ٣")

In [ ]:
# ═══════ الخلية ٣ — التوليد (٣٠ ثانية) ═══════
import numpy as np, time, os
from PIL import Image
from tsr.utils import remove_background, resize_foreground

img = Image.open(IMAGE).convert("RGBA")
if np.array(img)[:,:,3].max() == 0:
    try:
        import rembg
        img = rembg.remove(img, session=rembg.new_session())
        print("خلفية انشالت ✓")
    except Exception as e:
        print("rembg فشل (نكمل):", str(e)[:60])

proc = resize_foreground(remove_background(img), 0.85)
a = np.array(proc.convert("RGB")).astype(np.float32)/255.0
a = a * np.array(proc.convert("RGBA"))[:,:,3:4].astype(np.float32)/255.0

print("التوليد…")
t0 = time.time()
with torch.no_grad():
    codes = MODEL([torch.from_numpy(a[np.newaxis]).cuda()], device="cuda")
    mesh = MODEL.extract_mesh(codes, has_vertex_color=True, resolution=256)[0]
print(f"خلص بـ {time.time()-t0:.1f} ثانية")
print(f"رؤوس {len(mesh.vertices):,} · وجوه {len(mesh.faces):,}")

mesh.export("/content/model.glb")
mesh.export("/content/model.obj")
print(f"model.glb · {os.path.getsize('/content/model.glb')/1e6:.1f} MB")

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
V = np.asarray(mesh.vertices); F = np.asarray(mesh.faces)
if len(F) > 5000: F = F[np.random.choice(len(F), 5000, replace=False)]
fig = plt.figure(figsize=(15,5), facecolor="white")
for i,(el,az,nm) in enumerate([(18,35,"أمامي"),(0,0,"جانبي"),(75,20,"علوي")]):
    ax = fig.add_subplot(1,3,i+1, projection="3d")
    pc = Poly3DCollection(V[F], alpha=1.0); pc.set_facecolor("#4a90d9"); pc.set_edgecolor("none")
    ax.add_collection3d(pc); ax.auto_scale_xyz(V[:,0],V[:,1],V[:,2])
    ax.set_box_aspect((1,1,1)); ax.view_init(el,az); ax.set_axis_off(); ax.set_title(nm, fontsize=14)
plt.tight_layout()
plt.savefig("/content/samples.png", dpi=100, facecolor="white", bbox_inches="tight")
plt.show()
print("\n✅ روح للخلية ٤")

In [ ]:
# ═══════ الخلية ٤ — نزّل الملفات ═══════
import os, subprocess
subprocess.run("cd /content && zip -q -j ayso-3d.zip model.glb model.obj samples.png", shell=True)
print("الحجم:", round(os.path.getsize("/content/ayso-3d.zip")/1e6,1), "MB")
from google.colab import files
files.download("/content/ayso-3d.zip")
files.download("/content/model.glb")